# M1 — static lead-specific Gaussian copula

This notebook derives M1 without assuming another notebook. Read [Chapters 3--5](../../docs/scientific/03_chronos_and_pit.md) for PIT construction, dependence, and estimation.

## Notation, observation, and estimand

$i\in[N]$ indexes origin $t^{(i)}$, $g$ indexes the static set $\mathcal E_g=\{k_1,\ldots,k_{K_g}\}$, and $\tau\in[H]$ is one-based lead. $Y_{k,\tau}^{(i)}$ is random, $y_{k,\tau}^{(i)}$ is observed, $F_{k,\tau}^{(i)}$ is the fixed marginal, and $A_{g,\tau}^{(i)}=\sum_kY_{k,\tau}^{(i)}$ is the random aggregate. The observation is mapped to pseudo-PIT $u_{k,\tau}^{(i)}$ and score $z_{k,\tau}^{(i)}=\Phi^{-1}(u_{k,\tau}^{(i)})$. Define entity validity $V_{k,\tau}^{(i)}\in\{0,1\}$ and complete validity $V_{g,\tau}^{(i)}=\prod_{k\in\mathcal E_g}V_{k,\tau}^{(i)}$. The vector $\mathbf z_{g,\tau}^{(i)}\in\mathbb R^{K_g}$ exists only when this product is one. M1 estimates persistent same-lead association in these vectors.

## Estimator

For each lead, training vectors produce a Ledoit--Wolf covariance $\widehat\Sigma_{g,\tau}^{LW}$, a shrinkage estimator that moves the empirical covariance toward a well-conditioned structured target. It is normalized to
$$\widehat R_{ab}=\widehat\Sigma_{ab}/\sqrt{\widehat\Sigma_{aa}\widehat\Sigma_{bb}}.$$
After symmetric positive-definite stabilization, the same $\widehat R_{g,\tau}$ is used at every validation and test origin. M1 can represent positive and negative association, but not origin-dependent association.

In [ ]:
from pathlib import Path
import sys, numpy as np, matplotlib.pyplot as plt
PROJECT_ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'pyproject.toml').is_file())
if str(PROJECT_ROOT/'src') not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/'src'))
from simcast.config import load_base_config, load_method_config, resolve_run_config
from simcast.experiments import locate_compatible_cache
from simcast.fm.cache import load_pit_library
from simcast.dependence.static_gaussian import StaticGaussianCopula
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config
BASE_FILE='configs/bases/liander2024/transformer.yaml'
METHOD_FILE='configs/methods/m1_static_gaussian.yaml'
BASE_OVERRIDES=(); METHOD_OVERRIDES=()
FIT_IN_MEMORY=False; TRAIN_IF_MISSING=False; RUN_EVALUATION=False
RUN_DIR=PROJECT_ROOT/'runs'/'notebook_walkthrough'/'m1_static'
base=load_base_config(PROJECT_ROOT/BASE_FILE,overrides=BASE_OVERRIDES)
method=load_method_config(PROJECT_ROOT/METHOD_FILE,overrides=METHOD_OVERRIDES)
config=resolve_run_config(base,method)
cache_dir=locate_compatible_cache(base)
library=load_pit_library(cache_dir,access='training'); ds=library.dataset
entity_ids=[str(x) for x in ds.entity_id.values]; K_g=len(entity_ids)
print({'g':base.data.entity_type,'K_g':K_g,'share_across_leads':method.model.share_across_leads})


## Optional in-memory estimate for interpretation

This cell does not write an artifact. When `FIT_IN_MEMORY=True`, it fits exactly the M1 estimator to training-visible complete vectors and displays one lead. This is descriptive inspection, not a registered test evaluation.

In [ ]:
if FIT_IN_MEMORY:
    train=np.asarray(ds['split'].values)=='train'
    z=np.asarray(ds['pit_z'].values)[train]
    s=method.model
    model=StaticGaussianCopula(shrinkage=s.shrinkage,share_across_leads=s.share_across_leads,jitter=s.jitter).fit(z,entity_ids)
    R=np.asarray(model.correlation_matrix(1))
    fig,ax=plt.subplots(figsize=(6,5)); im=ax.imshow(R,vmin=-1,vmax=1,cmap='coolwarm')
    ax.set(title='M1 training estimate at lead 1',xlabel='entity',ylabel='entity'); fig.colorbar(im,ax=ax); plt.show()
else:
    print('Set FIT_IN_MEMORY=True to estimate M1 from training pseudo-scores.')


## Optional authoritative training and evaluation

These controls are false by default. The functions are identical to the CLI path and preserve cache validation, full-group assertions, saved metadata, and sealed test access.

In [ ]:
if TRAIN_IF_MISSING and not (RUN_DIR/'model.npz').is_file():
    train_from_config(config,cache_dir=cache_dir,output_dir=RUN_DIR)
if RUN_EVALUATION:
    evaluate_from_config(config,methods=('static_gaussian',),method_runs={'static_gaussian':RUN_DIR},cache_dir=cache_dir,output_dir=RUN_DIR/'evaluation')


## Interpretation and controls

`method.model.shrinkage` selects the covariance estimator; `share_across_leads: false` gives $H$ matrices and `true` pools all leads; `jitter` controls stabilization. M1 has no feature or optimization section because its estimator is closed-form. `StaticGaussianCopula` implements the estimate. A gain over M0 indicates persistent residual rank dependence. A conditional method's gain over M1 asks whether dependence varies predictably with origin context.